# Error de traslación de los loops — StereoLoopDetector

Para cada loop detectado (query `q`, match `m`) se compara la magnitud de la traslación estimada por PnP
(`loop_translation_x/y/z`, en el frame de la cámara) contra la distancia GPS planar entre ambos frames:

$$e = \left|\ \lVert t_{est} \rVert - \lVert p_{gps}[q] - p_{gps}[m] \rVert\ \right|$$

Se compara solo la magnitud porque el ground truth no tiene orientación y el detector descarta la rotación de PnP.


In [ ]:
from pathlib import Path
import numpy as np
import yaml
import matplotlib.pyplot as plt

RESULTS = Path('../build/fieldSAFE_dynamic_1_results.yml')
POSES = Path('/mnt/datalake/datasets/fieldsafe/prepared/poses_2016-10-25-11-41-21.csv')
STOPPED_PATH_M = 1.0  # si el robot recorrió menos que esto entre match y query, el loop es "con robot detenido"
print(RESULTS)

In [ ]:
def load_opencv_yml(path):
    # PyYAML no entiende la directiva %YAML:1.0 de OpenCV
    return yaml.safe_load(path.read_text().split('---', 1)[1])

res = load_opencv_yml(RESULTS)
q = np.array(res['loop_query_ids'], dtype=int)
m = np.array(res['loop_match_ids'], dtype=int)
t_est = np.stack([res['loop_translation_x'], res['loop_translation_y'], res['loop_translation_z']], axis=1)

poses = np.loadtxt(POSES, delimiter=',')
assert len(poses) == res['num_images'] and (poses[:, 0] == np.arange(len(poses))).all()
xy = poses[:, 1:3]
print(f"{res['num_images']} imágenes, {len(q)} loops")

In [ ]:
gt_dist = np.linalg.norm(xy[q] - xy[m], axis=1)
est_dist = np.linalg.norm(t_est, axis=1)
err = np.abs(est_dist - gt_dist)

cum_path = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(xy, axis=0), axis=1))])
path_between = cum_path[q] - cum_path[m]
stopped = path_between < STOPPED_PATH_M

def stats(name, e):
    print(f"{name:<28} n={len(e):4d}  media={e.mean():.3f} m  mediana={np.median(e):.3f} m  "
          f"p90={np.percentile(e, 90):.3f} m  max={e.max():.3f} m")

stats('Todos los loops', err)
stats('Robot en movimiento', err[~stopped])
stats('Robot detenido', err[stopped])

In [ ]:
groups = [('Todos los loops', err), ('Robot en movimiento', err[~stopped]), ('Robot detenido', err[stopped])]

fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
rng = np.random.default_rng(0)
box_color, point_color, ink, muted = '#3b6fb6', '#3b6fb6', '#1f2328', '#6b7280'

ax.boxplot([g for _, g in groups], widths=0.45, showfliers=False, patch_artist=True,
           boxprops=dict(facecolor='#dbe6f5', edgecolor=box_color, linewidth=1.5),
           medianprops=dict(color=ink, linewidth=2),
           whiskerprops=dict(color=box_color, linewidth=1.5),
           capprops=dict(color=box_color, linewidth=1.5))
for i, (_, g) in enumerate(groups, start=1):
    ax.scatter(i + rng.uniform(-0.15, 0.15, len(g)), g, s=10, color=point_color, alpha=0.35, linewidths=0, zorder=3)
    ax.scatter(i, g.mean(), marker='D', s=40, color='white', edgecolor=ink, linewidths=1.5, zorder=4)
    ax.annotate(f"media {g.mean():.3f} m\nmediana {np.median(g):.3f} m", (i + 0.27, np.median(g)),
                fontsize=8, color=ink, va='center')

ax.set_xticks(range(1, len(groups) + 1), [f"{n}\n(n={len(g)})" for n, g in groups])
ax.set_xlim(0.5, len(groups) + 0.95)
ax.set_ylabel('Error de traslación |‖t_est‖ − ‖Δp_gps‖|  [m]', color=ink)
ax.set_title(f"FieldSAFE dynamic #1 — error de traslación por query\n{RESULTS.name}", fontsize=11, color=ink)
ax.grid(axis='y', color='#e5e7eb', linewidth=0.8)
ax.set_axisbelow(True)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
for s in ('left', 'bottom'):
    ax.spines[s].set_color(muted)
ax.tick_params(colors=muted)
ax.text(0.99, 0.98, '◆ media   — mediana   · cada query', transform=ax.transAxes, ha='right', va='top', fontsize=8, color=muted)
fig.tight_layout()
fig.savefig('translation_error_boxplot.png')
plt.show()